# Inter-Annotator Agreement Analysis

Анализ согласованности двух аннотаторов (A1 и A2) на задаче ранжирования 5 вакансий для 30 резюме.

Исходные данные:
- Вакансии: 5_vacancies.csv
- Резюме: CV/1.docx - CV/30.docx
- Аннотации: annotations-for-the-first-30-vacancies.txt

Метрики:
- Top-K Accuracy (K = 1, 2, 3)
- Kendall tau
- Spearman rho
- Krippendorff alpha
- Cohen kappa (по Top-1)

## 1. Импорт библиотек

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import re
import os
import warnings
warnings.filterwarnings('ignore')

# Для Krippendorff alpha
try:
    import krippendorff
except ImportError:
    !pip install krippendorff
    import krippendorff

# Для Cohen kappa
from sklearn.metrics import cohen_kappa_score

print('Библиотеки загружены')

Библиотеки загружены


## 2. Загрузка данных

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# Путь к папке с данными в Google Drive
DATA_FOLDER = "/content/drive/MyDrive/vacancy-resume-matching-dataset"

ANNOTATIONS_PATH = os.path.join(DATA_FOLDER, 'annotations-for-the-first-30-vacancies.txt')
VACANCIES_PATH = os.path.join(DATA_FOLDER, '5_vacancies.csv')

print('Файл аннотаций:', 'найден' if os.path.exists(ANNOTATIONS_PATH) else 'НЕ НАЙДЕН')
print('Файл вакансий:', 'найден' if os.path.exists(VACANCIES_PATH) else 'НЕ НАЙДЕН')

Mounted at /content/drive
Файл аннотаций: найден
Файл вакансий: найден


In [3]:
# Парсинг файла аннотаций

def parse_annotations(filepath):
    """Парсит файл аннотаций. Возвращает списки рангов для A1 и A2."""
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()

    def extract_arrays(text):
        arrays = re.findall(r'\[([\d,\s]+)\]', text)
        result = []
        for arr in arrays:
            nums = [int(x.strip()) for x in arr.split(',') if x.strip()]
            if len(nums) == 5:
                result.append(nums)
        return result

    lines = content.split('\n')
    ann1_rankings = []
    ann2_rankings = []
    current_annotator = None

    for line in lines:
        if 'ANNOTATOR_1' in line:
            current_annotator = 1
        elif 'ANNOTATOR_2' in line:
            current_annotator = 2

        arrays = extract_arrays(line)
        if arrays:
            if current_annotator == 1:
                ann1_rankings.extend(arrays)
            elif current_annotator == 2:
                ann2_rankings.extend(arrays)

    return ann1_rankings, ann2_rankings


ann1_rankings, ann2_rankings = parse_annotations(ANNOTATIONS_PATH)

N_CVS = min(len(ann1_rankings), len(ann2_rankings))
N_VACANCIES = 5

print(f'Аннотатор 1: {len(ann1_rankings)} записей')
print(f'Аннотатор 2: {len(ann2_rankings)} записей')
print(f'Общее количество CV для анализа: {N_CVS}')

Аннотатор 1: 30 записей
Аннотатор 2: 30 записей
Общее количество CV для анализа: 30


In [4]:
# Формат данных:
# ann1_rankings[i] = [r1, r2, r3, r4, r5]
# где r_j = ранг вакансии j для i-го резюме (1 = лучшая, 5 = худшая)

print('Пример: резюме 1')
print(f'  A1: {ann1_rankings[0]}  (вакансия с рангом 1 -- лучшее совпадение)')
print(f'  A2: {ann2_rankings[0]}')
print()
print('Пример: резюме 2')
print(f'  A1: {ann1_rankings[1]}')
print(f'  A2: {ann2_rankings[1]}')

Пример: резюме 1
  A1: [2, 1, 4, 3, 5]  (вакансия с рангом 1 -- лучшее совпадение)
  A2: [4, 3, 1, 5, 2]

Пример: резюме 2
  A1: [1, 2, 3, 4, 5]
  A2: [2, 4, 3, 1, 5]


## 3. Вспомогательные функции

In [5]:
def ranks_to_ordering(ranks):
    """
    Преобразует вектор рангов в ordering (упорядоченный список вакансий).
    Вход: [3, 1, 2, 5, 4] -- ранги для вакансий V1..V5
    Выход: [2, 3, 1, 5, 4] -- вакансии в порядке убывания предпочтения
    """
    indexed = [(rank, vac_id) for vac_id, rank in enumerate(ranks, 1)]
    indexed.sort(key=lambda x: x[0])
    return [vac_id for _, vac_id in indexed]


def ordering_to_ranks(ordering):
    """Обратное преобразование: ordering -> ranks."""
    ranks = [0] * 5
    for rank, vac_id in enumerate(ordering, 1):
        if 1 <= vac_id <= 5:
            ranks[vac_id - 1] = rank
    return ranks


# Строим orderings для обоих аннотаторов
ann1_orderings = {}
ann2_orderings = {}

for i in range(N_CVS):
    cv_id = i + 1
    ann1_orderings[cv_id] = ranks_to_ordering(ann1_rankings[i])
    ann2_orderings[cv_id] = ranks_to_ordering(ann2_rankings[i])

print('Orderings построены')
print(f'Пример CV 1: A1 ordering = {ann1_orderings[1]}, A2 ordering = {ann2_orderings[1]}')

Orderings построены
Пример CV 1: A1 ordering = [2, 1, 4, 3, 5], A2 ordering = [3, 5, 2, 1, 4]


## 4. Расчет основных метрик IAA

In [6]:
# Top-K Accuracy: совпадает ли лучшая вакансия A1 с позициями 1..K у A2

top1_hits = 0
top2_hits = 0
top3_hits = 0

for cv_id in range(1, N_CVS + 1):
    a1_best = ann1_orderings[cv_id][0]  # лучшая вакансия по мнению A1
    a2_ordering = ann2_orderings[cv_id]

    if a1_best == a2_ordering[0]:
        top1_hits += 1
    if a1_best in a2_ordering[:2]:
        top2_hits += 1
    if a1_best in a2_ordering[:3]:
        top3_hits += 1

top1_acc = top1_hits / N_CVS
top2_acc = top2_hits / N_CVS
top3_acc = top3_hits / N_CVS

print(f'Top-1 Accuracy: {top1_acc:.1%} ({top1_hits} из {N_CVS})')
print(f'Top-2 Accuracy: {top2_acc:.1%} ({top2_hits} из {N_CVS})')
print(f'Top-3 Accuracy: {top3_acc:.1%} ({top3_hits} из {N_CVS})')
print()
print(f'Для сравнения, случайный baseline: Top-1 = 20%, Top-2 = 40%, Top-3 = 60%')

Top-1 Accuracy: 16.7% (5 из 30)
Top-2 Accuracy: 43.3% (13 из 30)
Top-3 Accuracy: 66.7% (20 из 30)

Для сравнения, случайный baseline: Top-1 = 20%, Top-2 = 40%, Top-3 = 60%


In [7]:
# Kendall tau и Spearman rho -- для каждого CV отдельно, потом среднее

kendall_taus = []
spearman_rhos = []

for cv_id in range(1, N_CVS + 1):
    r1 = ann1_rankings[cv_id - 1]
    r2 = ann2_rankings[cv_id - 1]

    tau, p_tau = stats.kendalltau(r1, r2)
    rho, p_rho = stats.spearmanr(r1, r2)

    if not np.isnan(tau):
        kendall_taus.append(tau)
    if not np.isnan(rho):
        spearman_rhos.append(rho)

mean_tau = np.mean(kendall_taus)
mean_rho = np.mean(spearman_rhos)
std_tau = np.std(kendall_taus)
std_rho = np.std(spearman_rhos)

print(f'Kendall tau:  mean = {mean_tau:+.3f}, std = {std_tau:.3f}')
print(f'Spearman rho: mean = {mean_rho:+.3f}, std = {std_rho:.3f}')
print()
print(f'Количество CV с рассчитанными метриками: {len(kendall_taus)}')

Kendall tau:  mean = +0.129, std = 0.463
Spearman rho: mean = +0.185, std = 0.517

Количество CV с рассчитанными метриками: 30


In [8]:
# Krippendorff alpha -- ордитальная шкала

# Формируем матрицу: 2 аннотатора x (N_CVS * N_VACANCIES) элементов
# Каждый элемент -- ранг, присвоенный данной вакансии для данного CV

reliability_data = np.zeros((2, N_CVS * N_VACANCIES))

for i in range(N_CVS):
    for j in range(N_VACANCIES):
        idx = i * N_VACANCIES + j
        reliability_data[0, idx] = ann1_rankings[i][j]
        reliability_data[1, idx] = ann2_rankings[i][j]

alpha_ordinal = krippendorff.alpha(
    reliability_data=reliability_data,
    level_of_measurement='ordinal'
)

alpha_nominal = krippendorff.alpha(
    reliability_data=reliability_data,
    level_of_measurement='nominal'
)

alpha_interval = krippendorff.alpha(
    reliability_data=reliability_data,
    level_of_measurement='interval'
)

print(f'Krippendorff alpha (ordinal):  {alpha_ordinal:.4f}')
print(f'Krippendorff alpha (nominal):  {alpha_nominal:.4f}')
print(f'Krippendorff alpha (interval): {alpha_interval:.4f}')
print()
print('Интерпретация alpha:')
print('  >= 0.80 -- хорошая надежность')
print('  0.67-0.80 -- допустимая')
print('  < 0.67 -- следует интерпретировать с осторожностью')

Krippendorff alpha (ordinal):  0.1884
Krippendorff alpha (nominal):  0.1362
Krippendorff alpha (interval): 0.1891

Интерпретация alpha:
  >= 0.80 -- хорошая надежность
  0.67-0.80 -- допустимая
  < 0.67 -- следует интерпретировать с осторожностью


In [9]:
# Cohen kappa -- только по Top-1 (номинальная метрика)

a1_top1 = [ann1_orderings[cv_id][0] for cv_id in range(1, N_CVS + 1)]
a2_top1 = [ann2_orderings[cv_id][0] for cv_id in range(1, N_CVS + 1)]

kappa = cohen_kappa_score(a1_top1, a2_top1)

print(f'Cohen kappa (по Top-1 выбору): {kappa:.4f}')
print()
print('Интерпретация kappa:')
print('  0.81-1.00 -- почти полное согласие')
print('  0.61-0.80 -- существенное')
print('  0.41-0.60 -- умеренное')
print('  0.21-0.40 -- слабое')
print('  < 0.20 -- незначительное')

Cohen kappa (по Top-1 выбору): -0.0593

Интерпретация kappa:
  0.81-1.00 -- почти полное согласие
  0.61-0.80 -- существенное
  0.41-0.60 -- умеренное
  0.21-0.40 -- слабое
  < 0.20 -- незначительное


## 5. Сводная таблица IAA

In [10]:
summary = pd.DataFrame({
    'Метрика': [
        'Top-1 Accuracy',
        'Top-2 Accuracy',
        'Top-3 Accuracy',
        'Kendall tau (mean)',
        'Spearman rho (mean)',
        'Krippendorff alpha (ordinal)',
        'Krippendorff alpha (nominal)',
        'Cohen kappa (Top-1)',
    ],
    'Значение': [
        f'{top1_acc:.1%}',
        f'{top2_acc:.1%}',
        f'{top3_acc:.1%}',
        f'{mean_tau:+.3f}',
        f'{mean_rho:+.3f}',
        f'{alpha_ordinal:.4f}',
        f'{alpha_nominal:.4f}',
        f'{kappa:.4f}',
    ],
    'Baseline (случайный)': [
        '20.0%',
        '40.0%',
        '60.0%',
        '0.000',
        '0.000',
        '0.000',
        '0.000',
        '0.000',
    ],
})

print(summary.to_string(index=False))

                     Метрика Значение Baseline (случайный)
              Top-1 Accuracy    16.7%                20.0%
              Top-2 Accuracy    43.3%                40.0%
              Top-3 Accuracy    66.7%                60.0%
          Kendall tau (mean)   +0.129                0.000
         Spearman rho (mean)   +0.185                0.000
Krippendorff alpha (ordinal)   0.1884                0.000
Krippendorff alpha (nominal)   0.1362                0.000
         Cohen kappa (Top-1)  -0.0593                0.000


## 6. Поэлементный анализ: согласие по каждому CV

In [11]:
# Для каждого CV считаем tau, rho и совпадение Top-1

per_cv_data = []

for cv_id in range(1, N_CVS + 1):
    r1 = ann1_rankings[cv_id - 1]
    r2 = ann2_rankings[cv_id - 1]

    tau, _ = stats.kendalltau(r1, r2)
    rho, _ = stats.spearmanr(r1, r2)

    a1_best = ann1_orderings[cv_id][0]
    a2_best = ann2_orderings[cv_id][0]
    top1_match = 1 if a1_best == a2_best else 0

    per_cv_data.append({
        'cv_id': cv_id,
        'a1_ranks': str(r1),
        'a2_ranks': str(r2),
        'a1_top1': a1_best,
        'a2_top1': a2_best,
        'top1_match': top1_match,
        'kendall_tau': tau,
        'spearman_rho': rho,
    })

per_cv_df = pd.DataFrame(per_cv_data)

print('CV с совпадением Top-1:')
print(per_cv_df[per_cv_df['top1_match'] == 1][['cv_id', 'a1_top1', 'a2_top1', 'kendall_tau', 'spearman_rho']].to_string(index=False))
print()
print('CV с наибольшим расхождением (tau < 0):')
worst = per_cv_df[per_cv_df['kendall_tau'] < 0].sort_values('kendall_tau')
print(worst[['cv_id', 'a1_top1', 'a2_top1', 'kendall_tau', 'spearman_rho']].to_string(index=False))

CV с совпадением Top-1:
 cv_id  a1_top1  a2_top1  kendall_tau  spearman_rho
     8        4        4     1.000000      1.000000
    24        1        1     0.400000      0.600000
    25        1        1     0.400000      0.500000
    28        1        1     0.737865      0.820783
    29        2        2     1.000000      1.000000

CV с наибольшим расхождением (tau < 0):
 cv_id  a1_top1  a2_top1  kendall_tau  spearman_rho
     5        1        2         -1.0          -1.0
     3        1        5         -0.8          -0.9
    27        5        3         -0.8          -0.9
     1        2        3         -0.2          -0.5
    12        1        4         -0.2           0.0
    13        1        4         -0.2           0.0
    15        2        4         -0.2           0.0


In [12]:
# Распределение Kendall tau по CV

print('Распределение Kendall tau:')
print(f'  min:    {per_cv_df["kendall_tau"].min():.3f}')
print(f'  Q1:     {per_cv_df["kendall_tau"].quantile(0.25):.3f}')
print(f'  median: {per_cv_df["kendall_tau"].median():.3f}')
print(f'  Q3:     {per_cv_df["kendall_tau"].quantile(0.75):.3f}')
print(f'  max:    {per_cv_df["kendall_tau"].max():.3f}')
print()

n_positive = (per_cv_df['kendall_tau'] > 0).sum()
n_zero = (per_cv_df['kendall_tau'] == 0).sum()
n_negative = (per_cv_df['kendall_tau'] < 0).sum()

print(f'tau > 0 (согласие):     {n_positive} из {N_CVS}')
print(f'tau = 0 (нет связи):    {n_zero} из {N_CVS}')
print(f'tau < 0 (расхождение):  {n_negative} из {N_CVS}')

Распределение Kendall tau:
  min:    -1.000
  Q1:     0.000
  median: 0.100
  Q3:     0.400
  max:    1.000

tau > 0 (согласие):     15 из 30
tau = 0 (нет связи):    8 из 30
tau < 0 (расхождение):  7 из 30


## 7. Анализ предпочтений аннотаторов по вакансиям

In [13]:
# Какие вакансии каждый аннотатор чаще ставит на первое место

vacancy_names = [
    'V1: Software Developer .Net',
    'V2: Remote Software Developer',
    'V3: Junior Level Software Developer',
    'V4: Backend Software Developer',
    'V5: Software Developer',
]

a1_top1_counts = pd.Series(a1_top1).value_counts().sort_index()
a2_top1_counts = pd.Series(a2_top1).value_counts().sort_index()

top1_dist = pd.DataFrame({
    'Вакансия': [vacancy_names[i-1] for i in range(1, 6)],
    'A1 (раз на 1-м месте)': [a1_top1_counts.get(i, 0) for i in range(1, 6)],
    'A2 (раз на 1-м месте)': [a2_top1_counts.get(i, 0) for i in range(1, 6)],
})

print('Распределение Top-1 выборов по вакансиям:')
print(top1_dist.to_string(index=False))

Распределение Top-1 выборов по вакансиям:
                           Вакансия  A1 (раз на 1-м месте)  A2 (раз на 1-м месте)
        V1: Software Developer .Net                     13                     10
      V2: Remote Software Developer                      9                      4
V3: Junior Level Software Developer                      6                      2
     V4: Backend Software Developer                      1                     11
             V5: Software Developer                      1                      3


In [14]:
# Средние ранги по вакансиям для каждого аннотатора

a1_mean_ranks = []
a2_mean_ranks = []

for vac_id in range(N_VACANCIES):
    a1_ranks_for_vac = [ann1_rankings[cv][vac_id] for cv in range(N_CVS)]
    a2_ranks_for_vac = [ann2_rankings[cv][vac_id] for cv in range(N_CVS)]
    a1_mean_ranks.append(np.mean(a1_ranks_for_vac))
    a2_mean_ranks.append(np.mean(a2_ranks_for_vac))

mean_rank_df = pd.DataFrame({
    'Вакансия': [vacancy_names[i] for i in range(5)],
    'A1 средний ранг': [f'{r:.2f}' for r in a1_mean_ranks],
    'A2 средний ранг': [f'{r:.2f}' for r in a2_mean_ranks],
    'Разница': [f'{abs(a-b):.2f}' for a, b in zip(a1_mean_ranks, a2_mean_ranks)],
})

print('Средние ранги по вакансиям (1 = лучший, 5 = худший):')
print(mean_rank_df.to_string(index=False))

Средние ранги по вакансиям (1 = лучший, 5 = худший):
                           Вакансия A1 средний ранг A2 средний ранг Разница
        V1: Software Developer .Net            2.00            2.73    0.73
      V2: Remote Software Developer            2.13            2.97    0.83
V3: Junior Level Software Developer            2.57            2.60    0.03
     V4: Backend Software Developer            3.73            2.80    0.93
             V5: Software Developer            4.53            3.90    0.63


## 8. Матрица совпадений Top-1

In [15]:
# Confusion matrix: A1 Top-1 vs A2 Top-1

confusion = np.zeros((N_VACANCIES, N_VACANCIES), dtype=int)

for cv_id in range(1, N_CVS + 1):
    a1_best = ann1_orderings[cv_id][0] - 1  # 0-indexed
    a2_best = ann2_orderings[cv_id][0] - 1
    confusion[a1_best, a2_best] += 1

confusion_df = pd.DataFrame(
    confusion,
    index=[f'A1: V{i+1}' for i in range(5)],
    columns=[f'A2: V{i+1}' for i in range(5)]
)

print('Матрица совпадений Top-1 (строки = A1, столбцы = A2):')
print(confusion_df.to_string())
print()
print(f'Совпадения на диагонали: {np.trace(confusion)} из {N_CVS}')

Матрица совпадений Top-1 (строки = A1, столбцы = A2):
        A2: V1  A2: V2  A2: V3  A2: V4  A2: V5
A1: V1       3       2       0       5       3
A1: V2       4       1       1       3       0
A1: V3       3       1       0       2       0
A1: V4       0       0       0       1       0
A1: V5       0       0       1       0       0

Совпадения на диагонали: 5 из 30


## 9. Полноранговое расхождение

In [16]:
# Среднее абсолютное отклонение рангов по позициям

rank_diffs = np.zeros((N_CVS, N_VACANCIES))

for i in range(N_CVS):
    for j in range(N_VACANCIES):
        rank_diffs[i, j] = abs(ann1_rankings[i][j] - ann2_rankings[i][j])

print('Среднее абсолютное отклонение рангов по вакансиям:')
for j in range(N_VACANCIES):
    mean_diff = np.mean(rank_diffs[:, j])
    print(f'  {vacancy_names[j]}: {mean_diff:.2f}')

print()
overall_mad = np.mean(rank_diffs)
print(f'Общее MAD по всем парам: {overall_mad:.2f}')
print(f'(при случайном назначении ожидаемое MAD ~ 1.6)')

Среднее абсолютное отклонение рангов по вакансиям:
  V1: Software Developer .Net: 1.53
  V2: Remote Software Developer: 1.37
  V3: Junior Level Software Developer: 1.03
  V4: Backend Software Developer: 1.60
  V5: Software Developer: 1.17

Общее MAD по всем парам: 1.34
(при случайном назначении ожидаемое MAD ~ 1.6)


In [17]:
# Распределение абсолютных отклонений

all_diffs = rank_diffs.flatten()
diff_counts = pd.Series(all_diffs).value_counts().sort_index()

print('Распределение абсолютных отклонений рангов:')
for diff_val, count in diff_counts.items():
    pct = count / len(all_diffs) * 100
    bar = '#' * int(pct / 2)
    print(f'  |r1 - r2| = {int(diff_val)}: {count:3d} ({pct:5.1f}%) {bar}')

exact_match_pct = (all_diffs == 0).sum() / len(all_diffs) * 100
close_match_pct = (all_diffs <= 1).sum() / len(all_diffs) * 100
print()
print(f'Точные совпадения (diff=0): {exact_match_pct:.1f}%')
print(f'Близкие совпадения (diff<=1): {close_match_pct:.1f}%')

Распределение абсолютных отклонений рангов:
  |r1 - r2| = 0:  46 ( 30.7%) ###############
  |r1 - r2| = 1:  46 ( 30.7%) ###############
  |r1 - r2| = 2:  27 ( 18.0%) #########
  |r1 - r2| = 3:  23 ( 15.3%) #######
  |r1 - r2| = 4:   8 (  5.3%) ##

Точные совпадения (diff=0): 30.7%
Близкие совпадения (diff<=1): 61.3%


## 10. Krippendorff alpha по отдельным вакансиям

In [18]:
# Для каждой вакансии отдельно: насколько согласованы аннотаторы
# в ранжировании именно этой вакансии

print('Krippendorff alpha (ordinal) по вакансиям:')
print()

for vac_id in range(N_VACANCIES):
    a1_ranks_for_vac = np.array([ann1_rankings[cv][vac_id] for cv in range(N_CVS)])
    a2_ranks_for_vac = np.array([ann2_rankings[cv][vac_id] for cv in range(N_CVS)])

    rel_data = np.vstack([a1_ranks_for_vac, a2_ranks_for_vac])
    alpha_vac = krippendorff.alpha(reliability_data=rel_data, level_of_measurement='ordinal')

    tau_vac, _ = stats.kendalltau(a1_ranks_for_vac, a2_ranks_for_vac)

    print(f'  {vacancy_names[vac_id]}')
    print(f'    alpha = {alpha_vac:.4f}, tau = {tau_vac:+.3f}')
    print()

Krippendorff alpha (ordinal) по вакансиям:

  V1: Software Developer .Net
    alpha = -0.1006, tau = -0.066

  V2: Remote Software Developer
    alpha = -0.0527, tau = +0.090

  V3: Junior Level Software Developer
    alpha = -0.0915, tau = -0.109

  V4: Backend Software Developer
    alpha = -0.0471, tau = +0.000

  V5: Software Developer
    alpha = 0.0298, tau = +0.051



## 11. Позиционное согласие

In [19]:
# Для каждой позиции в ранжировании: как часто аннотаторы согласны
# (совпадает ли вакансия, поставленная на k-е место)

print('Согласие по позициям в ранжировании:')
print()

for pos in range(N_VACANCIES):
    matches = 0
    for cv_id in range(1, N_CVS + 1):
        if ann1_orderings[cv_id][pos] == ann2_orderings[cv_id][pos]:
            matches += 1
    pct = matches / N_CVS * 100
    random_pct = 100 / N_VACANCIES
    print(f'  Позиция {pos + 1}: {matches}/{N_CVS} совпадений ({pct:.1f}%), случайный = {random_pct:.0f}%')

Согласие по позициям в ранжировании:

  Позиция 1: 5/30 совпадений (16.7%), случайный = 20%
  Позиция 2: 6/30 совпадений (20.0%), случайный = 20%
  Позиция 3: 11/30 совпадений (36.7%), случайный = 20%
  Позиция 4: 7/30 совпадений (23.3%), случайный = 20%
  Позиция 5: 16/30 совпадений (53.3%), случайный = 20%


## 12. Итоговый вывод

In [20]:
print('ИТОГОВЫЕ РЕЗУЛЬТАТЫ IAA')
print('=' * 60)
print()
print(f'Top-1 Accuracy:             {top1_acc:.1%}')
print(f'Top-2 Accuracy:             {top2_acc:.1%}')
print(f'Top-3 Accuracy:             {top3_acc:.1%}')
print(f'Kendall tau (mean +/- std): {mean_tau:+.3f} +/- {std_tau:.3f}')
print(f'Spearman rho (mean +/- std):{mean_rho:+.3f} +/- {std_rho:.3f}')
print(f'Krippendorff alpha (ord):   {alpha_ordinal:.4f}')
print(f'Cohen kappa (Top-1):        {kappa:.4f}')
print(f'Среднее |r1-r2|:            {overall_mad:.2f}')
print()
print('Согласованность аннотаторов низкая.')
print('Это означает, что задача ранжирования вакансий субъективна,')
print('и ожидать от модели точность выше 50% Top-1 нереалистично.')
print(f'Достигнутый потолок согласия: Top-1 = {top1_acc:.1%}.')

ИТОГОВЫЕ РЕЗУЛЬТАТЫ IAA

Top-1 Accuracy:             16.7%
Top-2 Accuracy:             43.3%
Top-3 Accuracy:             66.7%
Kendall tau (mean +/- std): +0.129 +/- 0.463
Spearman rho (mean +/- std):+0.185 +/- 0.517
Krippendorff alpha (ord):   0.1884
Cohen kappa (Top-1):        -0.0593
Среднее |r1-r2|:            1.34

Согласованность аннотаторов низкая.
Это означает, что задача ранжирования вакансий субъективна,
и ожидать от модели точность выше 50% Top-1 нереалистично.
Достигнутый потолок согласия: Top-1 = 16.7%.


In [21]:
# Экспорт результатов

per_cv_df.to_csv('iaa_per_cv.csv', index=False)
summary.to_csv('iaa_summary.csv', index=False)

print('Сохранено: iaa_per_cv.csv, iaa_summary.csv')

Сохранено: iaa_per_cv.csv, iaa_summary.csv
